In [1]:
import pandas as pd
import requests
import urllib3
from concurrent.futures import ThreadPoolExecutor, as_completed

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def check_url(url):
    if pd.isna(url) or str(url).strip() == "":
        return "EMPTY"

    headers = {
        "Referer": "https://roxiestreams.info/",
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36"
    }

    try:
        with requests.get(
            url,
            headers=headers,
            timeout=(5, 15),
            stream=True,
            allow_redirects=True,
            verify=False
        ) as response:

            if response.status_code == 200:
                chunk = next(response.iter_content(1024), None)
                if chunk:
                    return "WORKING"

            return "ERROR"

    except Exception:
        return "ERROR"


channel_list = ["main", "mainalt", "prem", "football", "usa", "now", "bein",
                "cbssn", "tudn", "tnt", "tnt1", "tnt2", "tnt3", "tnt4", "tnt5"]

first_list   = ["601", "602", "daffodil"]

second_list  = [
    "serverbandel.sbs",
    "lifesharper.com",
    "xn--1000-ugoa0hsb9a0hb.com",
    "833577.xyz",
    "securehelp.help"
]


# build URL list
rows = []
for channel in channel_list:
    for first in first_list:
        for second in second_list:
            url = f"https://{first}.{second}/{channel}.m3u8"
            rows.append({
                "channel": channel,
                "first": first,
                "second": second,
                "url": url
            })


def worker(row):
    row["status"] = check_url(row["url"])
    return row


results = []

with ThreadPoolExecutor(max_workers=30) as executor:
    futures = [executor.submit(worker, r) for r in rows]

    for f in as_completed(futures):
        results.append(f.result())


df = pd.DataFrame(results)

# print full dataframe
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

print(df.sort_values(by='channel')[['url', 'status']])

                                                           url   status
72                            https://601.833577.xyz/bein.m3u8    ERROR
73                            https://602.833577.xyz/bein.m3u8    ERROR
84                  https://daffodil.securehelp.help/bein.m3u8  WORKING
83                       https://daffodil.833577.xyz/bein.m3u8    ERROR
178                      https://601.lifesharper.com/bein.m3u8    ERROR
81                  https://daffodil.lifesharper.com/bein.m3u8  WORKING
179           https://601.xn--1000-ugoa0hsb9a0hb.com/bein.m3u8    ERROR
170                     https://601.serverbandel.sbs/bein.m3u8    ERROR
79                       https://602.securehelp.help/bein.m3u8    ERROR
78            https://602.xn--1000-ugoa0hsb9a0hb.com/bein.m3u8    ERROR
77                      https://602.serverbandel.sbs/bein.m3u8    ERROR
74       https://daffodil.xn--1000-ugoa0hsb9a0hb.com/bein.m3u8  WORKING
76                       https://602.lifesharper.com/bein.m3u8  